# 05 — Ablation: v5_arch + v6_long

04_diverse.ipynb 의 v4_diverse 위에서 architecture/objective + epoch ablation.

## Setup matrix

| run | epoch | λ_lb | λ_ortho | K_active | 결과 |
|---|---|---|---|---|---|
| v3_minimal | 30 | 0.1 | 0 | 13.88 | baseline |
| v4_diverse | 30 | 0.1 | 0 | 6.56 | corpus 다양화, source cluster |
| **v5_arch** | 30 | **0.05** | **0.05** | **1.00** | hard 1-hot, paradigm degenerate, 16 distinct domain |
| **v6_long** | 500 (チ79 멈춤) | 0.1 | 0 | 5.8 | long-train, plateau epoch ~15 |

**Pre-req**: 04_diverse 의 v4 학습 + cache 완료. 같은 corpus + fact_emb cache 재사용.

In [ ]:
import os, sys
BASE = '/content/drive/MyDrive/sideproject'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
os.chdir(BASE)
if BASE not in sys.path:
    sys.path.insert(0, BASE)
print('cwd =', os.getcwd())

## F. v5_arch — λ_ortho on (paradigm degenerate test)

λ_ortho 가 expert orthogonality 강화 → sparsegen λ saturating → K_active=1. 단 *cluster sharpness* 측면 진짜 evidence.

In [ ]:
from phase1.train import train_phase1

train_phase1(
    run_id='ph1_v5_arch',
    epochs=30,
    batch_size=32,
    lr=1e-4,
    warmup_steps=300,
    grad_clip=1.0,
    lambda_lb=0.05,
    lambda_ortho=0.05,
)

### F.1 — history + eval + cluster

In [ ]:
import json
from pathlib import Path
from phase1.eval import eval_phase1
from phase1.cluster_analysis import analyze_experts

run_id = 'ph1_v5_arch'
h = json.loads(Path(f'out/phase1/{run_id}/history.json').read_text())
first, last = h[0], h[-1]
print(f'[{run_id}] epoch {first["epoch"]} → {last["epoch"]}')
print(f'  loss        {first["loss"]:.4f} → {last["loss"]:.4f}')
print(f'  loss_recon  {first["loss_recon"]:.4f} → {last["loss_recon"]:.4f}')
print(f'  K_active    {first["active_frac"]*16:.2f} → {last["active_frac"]*16:.2f}  (of 16)')

result = eval_phase1(run_id=run_id)
print('\n=== summary ===')
print(f'mean_cosine: {result["cycle_reconstruction"]["mean_cosine"]:.4f}')
print(f'alpha macro_f1: {result["linear_probe_routed_alpha"].get("macro_f1"):.4f}')
print(f'subkg macro_f1: {result["linear_probe_sub_kg"].get("macro_f1"):.4f}')

analyze_experts(run_id)

### F.2 — SimBench eval

In [ ]:
from phase1.eval_simbench_classifier import train_simbench_classifier

train_simbench_classifier(
    run_id='ph1_v5_arch',
    epochs=100,
    batch_size=256,
    lr=1e-3,
    hidden=512,
    dropout=0.1,
)

In [ ]:
from phase1.cluster_analysis import compare_runs

compare_runs(['ph1_v3_minimal', 'ph1_v4_diverse', 'ph1_v5_arch'])

## G. v6_long — epoch 500 (under-trained 가설 verify)

v4 setting (λ_lb=0.1, λ_ortho=0) 의 long-train. K_active 가 6 유지하면서 cycle reconstruction + expert specialization 의 long-train 효과.

Self-supervised reference: BYOL/SimCLR 1000 epoch. 우리 update 수 측면 500 epoch 도 충분.

In [ ]:
from phase1.train import train_phase1

train_phase1(
    run_id='ph1_v6_long',
    epochs=500,
    batch_size=32,
    lr=1e-4,
    warmup_steps=300,
    grad_clip=1.0,
    lambda_lb=0.1,
    lambda_ortho=0.0,
)

### G.1 — Plateau detection

Loss_recon trajectory Š9 sliding-window mean → plateau 佋달 epoch. 30/50/100/150/200/250/300/500 checkpoint.

In [ ]:
import json
from pathlib import Path

run_id = 'ph1_v6_long'
h = json.loads(Path(f'out/phase1/{run_id}/history.json').read_text())
n = len(h)
print(f'[{run_id}] {n} epochs completed')

WIN = 30
recon = [e['loss_recon'] for e in h]
checkpoints = [30, 50, 100, 150, 200, 250, 300, 500, 1000]
print(f'\n{"epoch":>8} {"recon(win=30)":>15} {"K_active":>9} {"drop vs prev":>15}')
print('-' * 55)
prev_mean = None
for ep in checkpoints:
    if ep > n: break
    win = recon[max(0, ep-WIN):ep]
    mean = sum(win) / len(win)
    k_act = h[ep-1]['active_frac'] * 16
    if prev_mean is not None:
        drop_pct = (prev_mean - mean) / prev_mean * 100
        drop_str = f'{drop_pct:+.2f}%'
    else:
        drop_str = '—'
    print(f'{ep:>8} {mean:>15.4f} {k_act:>9.2f} {drop_str:>15}')
    prev_mean = mean

if n >= 20:
    last10 = sum(recon[-10:]) / 10
    prev10 = sum(recon[-20:-10]) / 10
    final_drop = (prev10 - last10) / prev10 * 100
    print(f'\n[final 10 epoch] drop {final_drop:+.3f}% — {"PLATEAU" if abs(final_drop) < 0.1 else "still learning"}')

### G.2 — eval + cluster + SimBench

In [ ]:
from phase1.eval import eval_phase1
from phase1.cluster_analysis import analyze_experts
from phase1.eval_simbench_classifier import train_simbench_classifier

run_id = 'ph1_v6_long'

result = eval_phase1(run_id=run_id)
print(f'\n=== {run_id} eval ===')
print(f'mean_cosine: {result["cycle_reconstruction"]["mean_cosine"]:.4f}')
print(f'alpha macro_f1: {result["linear_probe_routed_alpha"].get("macro_f1"):.4f}')
print(f'subkg macro_f1: {result["linear_probe_sub_kg"].get("macro_f1"):.4f}')

analyze_experts(run_id)

print(f'\n{"=" * 80}\nSimBench eval\n{"=" * 80}')
train_simbench_classifier(
    run_id=run_id, epochs=100, batch_size=256, lr=1e-3, hidden=512, dropout=0.1,
)

In [ ]:
from phase1.cluster_analysis import compare_runs

compare_runs(['ph1_v3_minimal', 'ph1_v4_diverse', 'ph1_v5_arch', 'ph1_v6_long'])